In [ ]:
import pandas as pd
import os

import sys
sys.path.append('..')
from helpers import get_dst_timestamps

In [2]:
isone_load_zone_subba_map = {
    'ME': '4001',
    'NH': '4002',
    'VT': '4003',
    'CT': '4004',
    'RI': '4005',
    'SEMA': '4006',
    'SEMASS': '4006',
    'WCMA': '4007',
    'WCMASS': '4007',
    'NEMA': '4008',
    'NEMASSBOST': '4008'
}

In [22]:
# Profiles downloaded from https://www.iso-ne.com/isoexpress/web/reports/load-and-demand/-/tree/zone-info
load_profile_list = []
forecast_profile_list = []

dir = '../data/iso_load_profiles/raw/isone'
for filename in os.listdir(dir):
    f = os.path.join(dir, filename)
    excel_file = pd.ExcelFile(f)
    isone_load_zone_subba_map_sub = {
        k:v for k,v in isone_load_zone_subba_map.items()
        if k in excel_file.sheet_names 
    }

    year = int(filename.split('_')[0])

    if year != 2024:
        continue

    dst_start, dst_end = get_dst_timestamps(year)    

    load_df_list = []
    forecast_df_list = []
    for load_zone, subba in isone_load_zone_subba_map_sub.items():
        df = excel_file.parse(load_zone)

        date_column = df.columns[0]
        assert date_column in [
            'Date'
        ], f"Date column is {date_column} for year {year}"
        
        hour_column = df.columns[1]
        assert hour_column in [
            'Hour',
            'Hr_End'
        ], f"Hour column is {hour_column} for year {year}"

        load_column = df.columns[3]
        assert load_column in [
            'DEMAND',
            'RT_Demand'
        ], f"Load column is {load_column} for year {year}"

        forecast_column = df.columns[2]
        assert forecast_column in [
            'DA_DEMD',
            'DA_Demand'
        ], f"Forecast column is {forecast_column} for year {year}"

        if year == 2024:
            # Note: This needs to be fixed to properly handle daylight savings
            df = df.loc[~df[hour_column].str.contains('X')]
            df['timestamp'] = df.apply(
                axis=1,
                func=lambda x: (
                    x[date_column] + pd.Timedelta(hours=pd.to_numeric(x[hour_column], errors='coerce'))
                )
            )
        else:
            df['timestamp'] = df.apply(
                axis=1,
                func=lambda x: x[date_column] + pd.Timedelta(hours=x[hour_column])
            )
        df = df[['timestamp', load_column, forecast_column]]

        # Convert daylight hours to standard by dropping the hour where
        # DST starts, moving daylight hours back by one hour, and filling in
        # the hour before DST ends with the DST-end value
        df = df.loc[df.timestamp != dst_start]
        df.loc[(
            (df.timestamp >= dst_start) & (df.timestamp < dst_end)
        ), 'timestamp'] -= pd.Timedelta(hours=1)

        # Convert from EST to UTC
        df["timestamp"] += pd.Timedelta(hours=5)
        df = df.sort_values("timestamp").assign(subba=subba)

        load_df = (
            df.rename(columns={load_column: 'value'})
            [['timestamp', 'subba', 'value']]
        )
        load_df_list.append(load_df)

        forecast_df = (
            df.rename(columns={forecast_column: 'value'})
            [['timestamp', 'subba', 'value']]
        )
        forecast_df_list.append(forecast_df)

    load_df = pd.concat(load_df_list, ignore_index=True)
    load_profile_list.append(load_df)
    
    forecast_df = pd.concat(forecast_df_list, ignore_index=True)
    forecast_profile_list.append(forecast_df)

In [ ]:
isone_load = pd.concat(load_profile_list, ignore_index=True)
isone_load.to_csv(f"../data/iso_load_profiles/isone.csv", index=False)

isone_forecast = pd.concat(forecast_profile_list, ignore_index=True)
isone_forecast.to_csv(f"../data/iso_load_profiles/isone_forecast.csv", index=False)